[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/08_frontier/08_arena_horizon.ipynb)

# 08 · Arena、time-horizon 与评测报告 — 动手复现

**MODULE 08 / 9** · 配套讲解：[08_讲解.html](./08_讲解.html) · 全程 **CPU**（纯 numpy/scipy，秒级运行，无下载）

本 notebook 把模块 08 的两条前沿线各自复现一遍：

| Part | 内容 | 对应论文 |
|---|---|---|
| **Part 1** | 合成对战数据 → 在线 Elo vs Bradley-Terry MLE → bootstrap 排名 CI | [Bradley & Terry 1952] [Chiang 2024] |
| **Part 2** | 合成任务套件 → logistic 拟合 success ~ log(human_time) → P50/P80 horizon → 代际增长图 | [Kwa 2025] |
| **练习** | ✏️ ×3：`elo_update` / `bt_nll` / `p50_horizon`，assert 自动判分 | — |

> ⚠️ 全部数据为**合成数据**：我们设定真值参数、生成观测、再用估计方法把真值"找回来"。
> 这是检验评测方法学的标准做法——真实世界里你永远不知道真值，但在仿真里方法的偏差与方差一目了然。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit, logit   # expit = sigmoid σ(x)

rng = np.random.default_rng(42)
print("numpy:", np.__version__)

## Part 1 · 从对战到排名：Elo、BT-MLE 与 bootstrap CI

**Bradley-Terry 模型**：每个模型有潜在强度 $\theta_i$，对战胜率只看强度差

$$P(i \succ j) = \sigma(\theta_i - \theta_j)$$

我们按这个模型**正向生成** 2000 场对战（5 个模型、真值 $\theta$ 已知），然后用两种方法**反向估计**：

1. **在线 Elo**：逐场更新 $R_a \leftarrow R_a + K(S_a - E_a)$ —— 等价于对 BT log-loss 做学习率为 $K$ 的在线 SGD，**结果依赖对战顺序**；
2. **BT MLE**：`scipy.optimize.minimize` 最小化 $-\sum_m \log\sigma(\theta_{w_m}-\theta_{l_m})$ —— 顺序无关、用满全部数据。

注意真值里第 2、3 名的 $\theta$ 只差 0.10——故意设近，等会儿用 bootstrap 看榜单上这俩的名次到底分不分得开。

In [ ]:
MODELS = ["Apex-72B", "Boreal-34B", "Cirrus-30B", "Dune-13B", "Ember-7B"]
theta_true = np.array([1.00, 0.35, 0.25, -0.45, -1.15])   # 和为 0（BT 只在平移意义下可辨识）
N_MODELS, N_BATTLES = len(MODELS), 2000

def simulate_battles(theta, n_battles, rng):
    # 返回 [(winner_idx, loser_idx), ...]，对战双方均匀随机抽取
    battles = []
    for _ in range(n_battles):
        i, j = rng.choice(len(theta), size=2, replace=False)
        if rng.random() < expit(theta[i] - theta[j]):
            battles.append((i, j))
        else:
            battles.append((j, i))
    return battles

battles = simulate_battles(theta_true, N_BATTLES, rng)
battles_arr = np.asarray(battles)

# 经验胜率矩阵：wins[i, j] = i 击败 j 的场数
wins = np.zeros((N_MODELS, N_MODELS))
for w, l in battles:
    wins[w, l] += 1
games = wins + wins.T
print(f"共 {N_BATTLES} 场 / {N_MODELS*(N_MODELS-1)//2} 个模型对，平均每对 ≈ {games.sum()/2/10:.0f} 场")
with np.printoptions(precision=2, suppress=True):
    print("经验胜率 P(行 胜 列):")
    print(np.where(games > 0, wins / np.maximum(games, 1), np.nan))

In [ ]:
def run_elo(battles, n_models, k=32, r_init=1000.0):
    # 在线 Elo：按对战发生顺序逐场更新（这正是它的弱点——顺序依赖）
    r = np.full(n_models, r_init)
    for w, l in battles:
        e_w = 1.0 / (1.0 + 10 ** ((r[l] - r[w]) / 400))   # base-10 / 400 刻度
        r[w] += k * (1 - e_w)
        r[l] += k * (0 - e_w)
    return r

elo = run_elo(battles, N_MODELS, k=32)
# 换到 BT 自然刻度便于和真值比较: theta = R · ln(10)/400，再中心化
theta_elo = (elo - elo.mean()) * np.log(10) / 400

print(f"{'模型':<12}{'Elo':>9}{'theta_elo':>11}{'theta_true':>12}")
for i in np.argsort(-elo):
    print(f"{MODELS[i]:<12}{elo[i]:>9.1f}{theta_elo[i]:>11.2f}{theta_true[i]:>12.2f}")
print("\n试一试：把 battles 逆序重放（battles[::-1]），Elo 评分会变 —— MLE 不会。")

In [ ]:
def bt_nll_ridge(theta, w_idx, l_idx, lam=1e-4):
    # BT 负对数似然（向量化 + 数值稳定: -logσ(x) = logaddexp(0, -x)）
    # 微小 ridge 解决平移不可辨识，固定解的位置
    diffs = theta[w_idx] - theta[l_idx]
    return np.logaddexp(0, -diffs).sum() + lam * np.sum(theta ** 2)

def fit_bt(battles_arr, n_models, x0=None):
    w_idx, l_idx = battles_arr[:, 0], battles_arr[:, 1]
    x0 = np.zeros(n_models) if x0 is None else x0
    res = minimize(bt_nll_ridge, x0, args=(w_idx, l_idx), method="BFGS")
    return res.x - res.x.mean()   # 中心化后与真值同一规范

theta_bt = fit_bt(battles_arr, N_MODELS)

rmse_elo = np.sqrt(np.mean((theta_elo - theta_true) ** 2))
rmse_bt  = np.sqrt(np.mean((theta_bt  - theta_true) ** 2))
print(f"{'模型':<12}{'theta_true':>11}{'BT-MLE':>9}{'Elo':>9}")
for i in np.argsort(-theta_true):
    print(f"{MODELS[i]:<12}{theta_true[i]:>11.2f}{theta_bt[i]:>9.2f}{theta_elo[i]:>9.2f}")
print(f"\nRMSE 对真值:  BT MLE = {rmse_bt:.3f}   在线 Elo = {rmse_elo:.3f}")
print("BT（顺序无关、批量 MLE）通常明显更准 —— Chatbot Arena 弃用在线 Elo 的原因 [Chiang 2024]")

### Bootstrap 排名置信区间

榜单给一个点估计排名是不够的：**名次本身是统计量，也有抽样不确定性**。
做法与模块 02 完全一致——把 2000 场对战记录**有放回重采样**，每个重采样集重新拟合 BT，
收集每个模型的 $\hat\theta$ 分布与名次分布，取分位数得到 95% CI [Chiang 2024]。

预期看到：第 1 名遥遥领先（CI 不重叠），但**第 2、3 名（真值只差 0.10）的 CI 大幅重叠**——
此时榜单写 "#2" 和 "#3" 纯属噪声排序，规范的报告应当视为并列。

In [ ]:
N_BOOT = 200   # 教学用 200 次足够；正式报告建议 ≥1000（CPU 上也只是十几秒的事）
boot_thetas = np.zeros((N_BOOT, N_MODELS))
boot_ranks  = np.zeros((N_BOOT, N_MODELS), dtype=int)

for b in range(N_BOOT):
    idx = rng.integers(0, N_BATTLES, size=N_BATTLES)        # 有放回重采样"投票记录"
    th = fit_bt(battles_arr[idx], N_MODELS, x0=theta_bt.copy())  # 热启动加速
    boot_thetas[b] = th
    boot_ranks[b, np.argsort(-th)] = np.arange(1, N_MODELS + 1)  # 名次: 1 = 最强

lo, hi = np.percentile(boot_thetas, [2.5, 97.5], axis=0)
print(f"{'模型':<12}{'theta_hat':>10}{'95% CI':>20}{'名次范围':>12}")
for i in np.argsort(-theta_bt):
    r_lo, r_hi = np.percentile(boot_ranks[:, i], [2.5, 97.5])
    print(f"{MODELS[i]:<12}{theta_bt[i]:>10.2f}   [{lo[i]:+.2f}, {hi[i]:+.2f}]"
          f"      #{int(r_lo)} - #{int(r_hi)}")

order = np.argsort(-theta_bt)
plt.figure(figsize=(6.2, 3.4))
plt.errorbar(np.arange(N_MODELS), theta_bt[order],
             yerr=[theta_bt[order] - lo[order], hi[order] - theta_bt[order]],
             fmt="o", capsize=4)
plt.xticks(np.arange(N_MODELS), [MODELS[i] for i in order], rotation=15)
plt.ylabel("BT strength (95% bootstrap CI)")
plt.title("Synthetic leaderboard with bootstrap CIs")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

p23 = (boot_thetas[:, 1] > boot_thetas[:, 2]).mean()
print(f"\nbootstrap 中 {MODELS[1]} 强于 {MODELS[2]} 的比例: {p23:.0%}")
print("该比例未达 97.5% ⇒ 第 2/3 名的名次差异在 95% 水平不显著，报告时应并列处理。")

## Part 2 · METR time-horizon 复现 [Kwa 2025]

方法四步（详见讲解 §4）：

1. **任务套件**：每个任务带人类专家完成时长标签 $t_k$（我们合成 30 个任务，$t_k$ 在 1–480 分钟**对数均匀**分布）；
2. **逐任务成败** $y_k\in\{0,1\}$：按一个隐藏的 logistic 真值模型生成（仿真里我们知道真值，可检验估计质量）；
3. **logistic 回归** $P(\text{success}) = \sigma(\alpha + \beta\log t)$，$\beta<0$ —— 这里用**手写牛顿法**（带 ridge 防完全分离）；
4. **反解 horizon**：

$$h_{50} = \exp\!\left(-\frac{\alpha}{\beta}\right), \qquad h_{p} = \exp\!\left(\frac{\mathrm{logit}(p)-\alpha}{\beta}\right)$$

注意 $\log h_{80} - \log h_{50} = \ln 4/\beta < 0$：**P80 恒短于 P50**，且比值只由斜率 $\beta$ 决定。

In [ ]:
def make_tasks(n_tasks, rng, t_min=1.0, t_max=480.0):
    # 人类完成时长: 1 到 480 分钟，对数均匀
    return np.exp(rng.uniform(np.log(t_min), np.log(t_max), size=n_tasks))

def true_success_prob(t_minutes, h50_true, beta_true=-1.1):
    # 隐藏真值: success ~ σ(β·(log t − log h50))
    return expit(beta_true * (np.log(t_minutes) - np.log(h50_true)))

def fit_logistic_newton(log_t, y, lam=1e-3, n_iter=50):
    # 手写牛顿法拟合 success ~ α + β·log t（ridge 防止完全分离时 MLE 发散）
    X = np.column_stack([np.ones_like(log_t), log_t])
    w = np.zeros(2)
    for _ in range(n_iter):
        p = expit(X @ w)
        g = X.T @ (p - y) + lam * w                                 # 梯度
        H = X.T @ (X * (p * (1 - p))[:, None]) + lam * np.eye(2)    # Hessian
        step = np.linalg.solve(H, g)
        w -= step
        if np.abs(step).max() < 1e-10:
            break
    return w[0], w[1]   # (α 截距, β 斜率)

N_TASKS, H50_TRUE = 30, 35.0   # 该合成"模型"的真实 P50 horizon = 35 分钟
t_tasks = make_tasks(N_TASKS, rng)
y_tasks = (rng.random(N_TASKS) < true_success_prob(t_tasks, H50_TRUE)).astype(float)

alpha_hat, beta_hat = fit_logistic_newton(np.log(t_tasks), y_tasks)
P50_hat = np.exp(-alpha_hat / beta_hat)
P80_hat = np.exp((logit(0.8) - alpha_hat) / beta_hat)
print(f"拟合: alpha={alpha_hat:.2f}, beta={beta_hat:.2f}")
print(f"P50 horizon = {P50_hat:.1f} min（真值 {H50_TRUE:.0f}）   P80 horizon = {P80_hat:.1f} min")
print("P80 远短于 P50：要 80% 可靠性，只能交给模型短得多的任务（部署视角 vs 能力视角）")

t_grid = np.logspace(0, np.log10(480), 200)
plt.figure(figsize=(6.8, 3.6))
plt.plot(t_grid, expit(alpha_hat + beta_hat * np.log(t_grid)), label="fitted logistic")
plt.plot(t_grid, true_success_prob(t_grid, H50_TRUE), "--", alpha=.6, label="hidden true curve")
plt.scatter(t_tasks, y_tasks + rng.normal(0, .015, N_TASKS), s=18, c="k", alpha=.5,
            label="tasks (success=1 / fail=0)")
plt.axvline(P50_hat, ls=":", c="tab:red");   plt.axhline(.5, ls=":", c="tab:red", alpha=.4)
plt.axvline(P80_hat, ls=":", c="tab:green"); plt.axhline(.8, ls=":", c="tab:green", alpha=.4)
plt.xscale("log"); plt.xlabel("human time-to-complete (min, log)"); plt.ylabel("P(success)")
plt.title(f"Time-horizon fit:  P50={P50_hat:.0f} min,  P80={P80_hat:.0f} min")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 3 个能力递增的合成"模型代际" —— 纯合成数据，仅演示方法论。
# [Kwa 2025] 的真实测量: 2019–2025 前沿模型 P50 horizon 约每 7 个月翻倍。
GENERATIONS = {"gen-1 (2023)": 4.0, "gen-2 (2024)": 15.0, "gen-3 (2025)": 60.0}  # 真值 h50 (min)
N_TASKS_G = 60   # 每代多给些任务，拟合更稳

p50s = []
for name, h50 in GENERATIONS.items():
    t = make_tasks(N_TASKS_G, rng)
    y = (rng.random(N_TASKS_G) < true_success_prob(t, h50)).astype(float)
    a, b = fit_logistic_newton(np.log(t), y)
    p50s.append(np.exp(-a / b))
    print(f"{name}:  真值 h50 = {h50:>5.0f} min   拟合 P50 = {p50s[-1]:6.1f} min")

plt.figure(figsize=(5.6, 3.4))
plt.semilogy(range(len(p50s)), p50s, "o-")
plt.xticks(range(len(p50s)), list(GENERATIONS))
plt.ylabel("P50 horizon (min, log scale)")
plt.title("Horizon grows ~exponentially across generations\n(synthetic demo of [Kwa 2025] methodology)")
plt.grid(True, which="both", alpha=.3)
plt.tight_layout(); plt.show()
print("半对数图上近似直线 = 指数增长。真实版的争议（任务分布、外推合法性）见讲解 §4–5。")

## ✏️ 练习 1：实现 `elo_update`

实现单场 Elo 更新 `elo_update(r_a, r_b, score_a, k=32)`，返回 `(new_r_a, new_r_b)`。

- 预期得分 $E_a = \dfrac{1}{1+10^{(R_b-R_a)/400}}$；更新 $R_a \leftarrow R_a + K(S_a - E_a)$；
- B 侧对称：$S_b = 1 - S_a$，$E_b = 1 - E_a$；
- 性质提示：更新是**零和**的（$\Delta R_a = -\Delta R_b$），等分对战胜者恰得 $+K/2$。

10 行以内可完成。

In [ ]:
def elo_update(r_a, r_b, score_a, k=32):
    '''单场 Elo 更新。score_a: A 的得分 (1=胜, 0.5=平, 0=负)。返回 (new_r_a, new_r_b)。'''
    # TODO: 1) e_a = 1 / (1 + 10**((r_b - r_a)/400))
    # TODO: 2) new_r_a = r_a + k * (score_a - e_a)
    # TODO: 3) B 侧对称更新 (S_b = 1-score_a, E_b = 1-e_a)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ra, rb = elo_update(1000, 1000, 1.0)
assert abs(ra - 1016) < 1e-9 and abs(rb - 984) < 1e-9, "等分对战胜者应 +K/2 = +16"
ra, rb = elo_update(1200, 1000, 1.0, k=32)
assert abs((ra + rb) - 2200) < 1e-9, "Elo 更新应零和（总分守恒）"
assert abs(ra - 1207.6881) < 1e-3, f"已知数值: 期望 1207.6881, 得到 {ra:.4f}"
# 对称性: 交换 A/B 视角应得到一致结果
ra1, rb1 = elo_update(1100, 900, 0.0)
rb2, ra2 = elo_update(900, 1100, 1.0)
assert abs(ra1 - ra2) < 1e-9 and abs(rb1 - rb2) < 1e-9, "交换 A/B 应等价"
ra, rb = elo_update(1000, 1000, 0.5)
assert ra == 1000 and rb == 1000, "同分平局应不变"
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `bt_nll`（可直接喂给 optimizer）

实现 Bradley-Terry 负对数似然 `bt_nll(theta, battles)`：

$$\mathcal{L}(\theta) = -\sum_{(w,l)\in\text{battles}} \log\sigma(\theta_w - \theta_l)$$

- `theta`: shape `(n_models,)`；`battles`: `[(winner_idx, loser_idx), ...]`；返回标量；
- 数值稳定提示：$-\log\sigma(x) = \log(1+e^{-x})$ = `np.logaddexp(0, -x)`；
- 写好后它就是 `scipy.optimize.minimize` 的目标函数——正文 `fit_bt` 的核心就这几行。

自测会用**数值差分**检查梯度方向：模型 0 战绩更好时 $\partial\mathcal{L}/\partial\theta_0 < 0$（增大 $\theta_0$ 降低损失），并对照解析值 $-(\text{实际胜场}-\text{期望胜场})$。

In [ ]:
def bt_nll(theta, battles):
    '''Bradley-Terry 负对数似然: -Σ log σ(θ_w − θ_l)。返回标量。'''
    theta = np.asarray(theta)
    # TODO: 取出每场的 diff = θ_winner − θ_loser（提示: 整数数组索引）
    # TODO: 返回 np.logaddexp(0, -diff).sum()
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
toy = [(0, 1), (0, 1), (1, 0)]   # 模型 0 胜 2 负 1
# θ=0 时每场胜率 0.5 → NLL = 3·ln2
assert np.isclose(bt_nll(np.zeros(2), toy), 3 * np.log(2)), "θ=0 时 NLL 应为 3·ln2"
# 模型 0 战绩更好 → θ=[1,0] 似然更高（NLL 更低）
assert bt_nll(np.array([1.0, 0.0]), toy) < bt_nll(np.array([0.0, 1.0]), toy)
# 数值梯度方向 + 解析值对照: ∂NLL/∂θ_0|_(θ=0) = -(2 - 1.5) = -0.5
eps = 1e-5
g0 = (bt_nll(np.array([eps, 0.0]), toy) - bt_nll(np.array([-eps, 0.0]), toy)) / (2 * eps)
assert g0 < 0, f"∂NLL/∂θ_0 应为负, 得到 {g0:.4f}"
assert np.isclose(g0, -0.5, atol=1e-6), f"解析梯度应为 -0.5, 数值差分得到 {g0:.6f}"
# 直接喂给 scipy:
res = minimize(bt_nll, np.zeros(2), args=(toy,), method="BFGS")
assert res.x[0] > res.x[1], "拟合后模型 0 应更强"
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `p50_horizon`（解析反解）

从 logistic 参数解析解出 P50 horizon：`p50_horizon(coef, intercept)`。

- 模型：$P(\text{success}) = \sigma(\text{intercept} + \text{coef}\cdot\log t)$，`coef` < 0；
- $\sigma(z)=\tfrac12 \iff z=0$，在 **log 空间**反解 $\log h_{50}$ 再取 `np.exp`；
- 一行可完成。自测会对照正文拟合出的 `P50_hat`，并验证解出的点上预测成功率恰为 0.5。

In [ ]:
def p50_horizon(coef, intercept):
    '''由 P(success)=σ(intercept + coef·log t) 解析解出 P50 horizon（与 t 同单位）。'''
    # TODO: intercept + coef·log h50 = 0  →  h50 = ?
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 已知参数: β=-1, α=log(30) → h50 恰为 30
assert np.isclose(p50_horizon(-1.0, np.log(30.0)), 30.0)
# P50 只由 -α/β 决定: (α, β) 同乘常数不变
assert np.isclose(p50_horizon(-2.0, 2 * np.log(30.0)), 30.0)
# 对照正文 Part 2 的拟合结果
assert np.isclose(p50_horizon(beta_hat, alpha_hat), P50_hat), "应与正文 P50_hat 一致"
# 自洽性: 在解出的 h50 处，logistic 预测成功率应恰为 0.5
h = p50_horizon(beta_hat, alpha_hat)
assert abs(expit(alpha_hat + beta_hat * np.log(h)) - 0.5) < 1e-9
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。三题合计不到 20 行——前沿评测的统计内核其实非常小，难的是知道**该算什么、怎么报**。

In [ ]:
# 练习 1 参考实现（先自己做，再对照）
def elo_update(r_a, r_b, score_a, k=32):
    e_a = 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400))
    new_r_a = r_a + k * (score_a - e_a)
    new_r_b = r_b + k * ((1 - score_a) - (1 - e_a))   # 零和: Δb = -Δa
    return new_r_a, new_r_b

In [ ]:
# 练习 2 参考实现（先自己做，再对照）
def bt_nll(theta, battles):
    theta = np.asarray(theta)
    w = np.array([b[0] for b in battles])
    l = np.array([b[1] for b in battles])
    return np.logaddexp(0, -(theta[w] - theta[l])).sum()

In [ ]:
# 练习 3 参考实现（先自己做，再对照）
def p50_horizon(coef, intercept):
    return np.exp(-intercept / coef)

## 全课总结 · 9 个模块一句话脉络

| # | 模块 | 一句话 |
|---|---|---|
| 00 | 总览与环境 | 评测是科学实验，不是跑分仪式 |
| 01 | 分类学与全景 | 先想清楚测的是能力还是对齐，基准有生命周期 |
| 02 | 统计严谨性 | 没有误差棒的分数不是测量 [Miller 2024] |
| 03 | 抽取与敏感性 | 分差可能只是模板措辞与正则抽取的运气 [Sclar 2023] |
| 04 | LLM-as-a-Judge | judge 便宜但有偏，先校准偏置再上岗 [Zheng 2023] |
| 05 | 污染与饱和 | 先排除"它是不是背过题" [Oren 2023] |
| 06 | 能力引出 | 引出不足只能测到下界，pass@k 要用无偏估计 [Chen 2021] |
| 07 | Harness 工程 | 可复现 = 配置即数据，一键复跑 [Biderman 2024] |
| 08 | Arena 与 horizon | 排名要带 bootstrap CI，能力要有物理单位 [Chiang 2024; Kwa 2025] |

把这九句连起来就是一条完整的工作流：**选对任务 → 设计好统计 → 控制好抽取与判分 → 排除污染 → 引出充分 → 工程可复现 → 用对排名/刻度 → 透明报告**。

---

🎉 **恭喜完成《LLM 评测科学》全部 9 个模块！**

你现在拥有的不只是一堆基准的名字，而是一套对任何评测结果做"质检"的方法论。
回到 [课程主页](../index.html) 查看全课地图、术语词典与论文清单——以及下一步的延伸方向。

---
## 🎯 真实数据胶囊题：真实成对结果上的 Elo 评分（Arena 内核）

Chatbot Arena 用 Elo 把两两胜负变成排行榜。用真实红酒质量等级当“选手”、真实质量差生成对战，实现在线 Elo 更新，验证恢复的 Elo 排名与真实质量一致。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

import pandas as pd
p=_f("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
df=pd.read_csv(p,sep=";"); levels=sorted(df["quality"].unique()); idx={q:i for i,q in enumerate(levels)}
rng=np.random.default_rng(0)
battles=[]
for _ in range(5000):
    a,b=rng.choice(levels,2,replace=False)
    win_a = rng.random() < 1/(1+10**(-(a-b)/2))   # 真实质量差 -> 胜率(Elo式)
    battles.append((idx[a], idx[b], 1 if win_a else 0))
print(f"{len(battles)} 场对战, {len(levels)} 个质量等级")

**练习**：实现 `fit_elo(battles, n, K, base)`：每场按 Elo 规则更新 `E=1/(1+10^((Rj-Ri)/400))`，`Ri += K*(win_i - E)`。返回最终评分向量。

In [ ]:
def fit_elo(battles, n, K=16, base=1000.0):
    # TODO: R=full(n,base); 每场 (i,j,win_i): 期望 Ei=1/(1+10**((R[j]-R[i])/400)); R[i]+=K*(win_i-Ei); R[j]+=K*((1-win_i)-(1-Ei))
    raise NotImplementedError


In [ ]:
# 自测：Elo 排名应与真实质量一致
R=fit_elo(battles, len(levels), K=16)
assert len(R)==len(levels)
assert list(np.argsort(R))==list(range(len(levels))), f"Elo 应随质量单调, 得到 {R.round(0)}"
print(f"恢复的 Elo(按质量排) = {R.round(0)} ✓ 单调递增")


### 📖 参考答案

In [ ]:
def fit_elo(battles, n, K=16, base=1000.0):
    R=np.full(n, base, float)
    for i,j,wi in battles:
        Ei=1/(1+10**((R[j]-R[i])/400))
        R[i]+=K*(wi-Ei); R[j]+=K*((1-wi)-(1-Ei))
    return R
print("✓ Arena 排行榜的内核就是这几行 Elo 更新")